# x7_custo-soja — Custos de Produção de Soja CONAB (Série Histórica MT)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** custos de produção de soja, unidade monetária e base física a conferir por aba, localidades de MT, safra/ano*  
**Conceito e Mecanismo Econômico:** Estrutura de custos operacionais (insumos, maquinário, mão de obra) de soja em praças selecionadas de MT. Fonte fundamental para o retorno líquido da terra.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__costs_conab__serie-historica-custos-soja-1997-a-2026__MT_celulas.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Extração de células, não série pronta; identificar R$/ha ou R$/unidade, local, tecnologia e categoria de custo.

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`arquivo_origem`**: Nome do arquivo original CONAB.
- **`aba`**: Nome da aba da planilha (identifica município e ano/safra).
- **`linha_excel`**: Linha original.
- **`coluna_excel`**: Coluna original.
- **`conteudo`**: Dado bruto da célula.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__costs_conab__serie-historica-custos-soja-1997-a-2026__MT_celulas.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x7_custo-soja`:
- **Atenção:** Extração de células, não série pronta; identificar R$/ha ou R$/unidade, local, tecnologia e categoria de custo.


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
# REGRA DO TXT: 'Extração de células, não série pronta; identificar R$/ha ou R$/unidade, local, tecnologia e categoria de custo.'
print(f"Abas disponíveis na série de custos de soja: {df['aba'].nunique()}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print(df.head(10))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
df['aba'].value_counts().head(10).plot(kind='barh')
plt.title("Top Abas por Número de Células")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df['ano_estimado'] = df['aba'].str.extract(r'(\d{4})')[0]
print("Contagem por ano estimado na aba:")
print(df['ano_estimado'].value_counts().sort_index())

## Fase 6: Feature Engineering — Custos de Produção de Soja CONAB

### 1. Step 1: Harmonização da Safra de Custos
- Periodicidade por safra agrícola CONAB.
- Métricas: Custo Operacional Efetivo (COE - desembolso direto), Custo Operacional Total (COT - inclui depreciação de máquinas) e Custo Total (CT) em R$/ha e R$/saca de 60 kg.

### 2. Step 2: Margem Bruta Operacional e RESSALVA METODOLÓGICA CRUCIAL
- **Margem Bruta Estimada por Saca:** $Margem_{saca} = PrecoRecebido - COE_{saca}$.
- **RESSALVA ECONOMÉTRICA FUNDAMENTAL:**
  A base da CONAB pesquisa apenas **2 a 4 municípios-polo** em Mato Grosso (ex: Sorriso, Campo Verde, Primavera do Leste e Sapezal).
  **NÃO IMPUTAR** esses custos para os outros 137 municípios no painel principal! A imputação sintética criaria variação espacial espúria e viés de atenuação severo por erro de medição. Esses custos devem ser usados exclusivamente em análises de robustez no subsample de polos.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Estruturação dos Custos Operacionais da Soja
df_custo = df.copy()

# Identificação das variáveis de custo por hectare e por saca de 60 kg
# coe_ha: custo operacional efetivo por hectare (R$/ha) -> desembolso financeiro direto
# cot_ha: custo operacional total por hectare (R$/ha) -> inclui depreciação de maquinário
# ct_ha: custo total por hectare (R$/ha) -> inclui remuneração esperada da terra e do capital
print("Métricas de Custo CONAB disponíveis:")
print(df_custo.head())

# 2. Defasagem temporal de 1 ano (t-1)
# Os custos planejados e observados na safra anterior orientam o orçamento da safra presente
print("\nNOTA METODOLÓGICA: Dados restritos a municípios-polo da CONAB. Não imputar aos 141 municípios.")